![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# MLOps Essentials
*Data Science & Machine Learning - Week 8, Day 2*

## How to use this notebook

In Project 2 you trained a model, saved it and demoed it. Now imagine that model in a real company. Six months later someone asks: *Which version is in production? What data was it trained on? Can you retrain it? Why is it worse than last spring?* If the answers are "on my laptop", "I don't remember" and "it worked on my machine", the model is not a product yet.

**MLOps** (machine learning operations) is the set of practices that turn a notebook into something a team can trust, reproduce and maintain. You will:

1. See where a model fits in a real ML system, and what goes wrong without MLOps.
2. Make your work **reproducible**: virtual environments and a pinned `requirements.txt`.
3. **Save** a trained pipeline with its metadata, and load it again.
4. **Track experiments** with MLflow: parameters, metrics, tags and models, and compare runs.
5. **Register** the best model and manage its versions with aliases, then load "the production model" by name.
6. Use the **DEV to PROD** workflow: branches, pull requests and a check that runs before every merge.
7. **Monitor** a model for drift.

We use the telecom churn data from Weeks 4 and 5 (`data/churn.csv`), so the models will look familiar. The slides **01. and 03. Development Environments** in this folder go with sections 2 and 6, and **6. Introduction to MLOps** is a longer read about the ideas of section 1. The **Appendix** has extra material that is not part of the lesson.

## Table of contents

- [Setup](#Setup)
- [1. Why MLOps?](#1.-Why-MLOps?)
- [2. Reproducible environments](#2.-Reproducible-environments)
- [3. Saving and loading a pipeline](#3.-Saving-and-loading-a-pipeline)
- [4. Tracking experiments with MLflow](#4.-Tracking-experiments-with-MLflow)
- [5. The Model Registry](#5.-The-Model-Registry)
- [6. From DEV to PROD](#6.-From-DEV-to-PROD)
- [7. Monitoring and drift](#7.-Monitoring-and-drift)
- [8. MLOps responsibly](#8.-MLOps-responsibly)
- [Summary](#Summary)
- [Appendix](#Appendix)

## Setup

You need `pip install mlflow scikit-learn pandas scipy joblib`. MLflow is the only new library: it is open source, runs on your computer, and needs no account. The notebook creates a small database (`mlflow.db`) and a folder (`mlruns`) next to it; both are generated data, so they belong in `.gitignore`.

In [1]:
import os
import sys
import json
import logging
import warnings
from datetime import datetime, timezone
from importlib.metadata import version

os.environ["MLFLOW_LOGGING_LEVEL"] = "WARNING"      # keep MLflow's info messages out of the notebook
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

import joblib
import numpy as np
import pandas as pd
import mlflow
import skops.io
from mlflow import MlflowClient
from mlflow.models import infer_signature
from scipy.stats import ks_2samp
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("Python", sys.version.split()[0], "| scikit-learn", version("scikit-learn"), "| MLflow", version("mlflow"))

Python 3.12.4 | scikit-learn 1.9.1 | MLflow 3.17.2


## 1. Why MLOps?

A famous Google paper, *Hidden Technical Debt in Machine Learning Systems* (2015), drew a real ML system as a big diagram in which the model code is a tiny box. Around it: collecting and checking data, extracting features, configuration, serving the predictions, monitoring them, and the infrastructure that holds it all together.

[visual: a small box "ML code" in the middle of large boxes: data collection, data verification, feature extraction, configuration, serving infrastructure, monitoring, analysis tools, process management]

Here is what goes wrong when only the small box gets attention, and the practice that fixes each problem:

| Problem | Practice | Section |
|---|---|---|
| "It works on my machine" | a virtual environment and a pinned `requirements.txt` | 2 |
| The model only exists inside a notebook | save the whole pipeline, with its metadata | 3 |
| "Which settings gave us 0.92?" | track every experiment | 4 |
| "Which model is in production?" | a model registry with versions and aliases | 5 |
| A broken change reaches users | branches, pull requests and checks before merging | 6 |
| The model gets worse and nobody notices | monitoring and retraining | 7 |

**MLOps borrows from DevOps** (the practices software teams use to ship code safely) and adds what is special about ML: the behaviour of a model depends on data, and data changes.

## 2. Reproducible environments

Your code depends on Python and on dozens of libraries, and each of them changes from version to version. A model saved with one version of scikit-learn may not load with another, and a pandas default may change between releases. To reproduce a result, you need the **same environment**.

### One virtual environment per project

A **virtual environment** is a folder with its own Python and its own libraries, separate from the rest of your computer. You create one per project, in a terminal:

```bash
python -m venv .venv                  # create it (once per project)
source .venv/bin/activate             # activate it (Windows: .venv\Scripts\activate)
pip install pandas scikit-learn mlflow
pip freeze > requirements.txt         # write down every installed version
deactivate                            # leave it
```

With conda, the equivalent is `conda create -n churn python=3.12`, `conda activate churn` and `conda env export > environment.yml`.

### What goes in git

**Commit the `requirements.txt`, never the `.venv` folder.** The folder is large, it contains files that only work on your operating system, and anyone can rebuild it in one command:

```bash
python -m venv .venv && source .venv/bin/activate && pip install -r requirements.txt
```

`pip freeze` lists everything, including libraries you never imported yourself. A shorter file with only your **direct** dependencies, each pinned to an exact version, is easier to read; here is how to write one for this notebook:

In [2]:
packages = ["pandas", "numpy", "scikit-learn", "mlflow", "scipy", "joblib"]
requirements = "\n".join(f"{name}=={version(name)}" for name in packages)
with open("requirements-churn.txt", "w") as file:
    file.write(requirements + "\n")
print(requirements)

pandas==3.1.0
numpy==2.5.4
scikit-learn==1.9.1
mlflow==3.17.2
scipy==1.18.1
joblib==1.6.0


`==` pins an exact version. You will also see `>=` (at least this version), which is more flexible and less reproducible. Add the Python version to your README, because it is not part of the file. The lab practises exactly this: your partner must be able to rebuild your environment from your repository alone.

## 3. Saving and loading a pipeline

First, a model to work with: the churn data, the same split as always, and a pipeline that prepares the columns and fits a random forest. Two small changes from Week 5 will matter later: the identifier columns (phone, state, area code) are dropped, and the numeric columns are stored as floats, which is what MLflow expects when values can be missing.

In [3]:
churn = pd.read_csv("data/churn.csv")
y = (churn["Churn?"] == "True.").astype(int)
X = churn.drop(columns=["Churn?", "Phone", "Area Code", "State"])
X = X.astype({column: "float64" for column in X.select_dtypes("number").columns})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

categorical = ["Int'l Plan", "VMail Plan"]
numeric = [column for column in X.columns if column not in categorical]
preprocessing = ColumnTransformer([("categorical", OneHotEncoder(drop="if_binary"), categorical),
                                   ("numeric", StandardScaler(), numeric)])

def make_pipeline(model):
    return Pipeline([("preprocessing", preprocessing), ("model", model)])

pipeline = make_pipeline(RandomForestClassifier(n_estimators=200, random_state=42)).fit(X_train, y_train)
print(f"{X_train.shape[0]} training rows, {X_train.shape[1]} features, churn rate {y_train.mean():.1%}")
print(f"Test ROC AUC: {roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1]):.3f}")

2666 training rows, 17 features, churn rate 14.5%
Test ROC AUC: 0.888


Save the **whole pipeline**, not just the model: the scaler and the encoder are part of it, and a model without them would receive unscaled, unencoded data. `joblib` is scikit-learn's recommended way to save and load objects. Next to the file, save **metadata**: what the model is, when and how it was trained, with which library versions, on which columns, and how good it was.

In [4]:
os.makedirs("saved_models", exist_ok=True)
joblib.dump(pipeline, "saved_models/churn_pipeline.joblib")

metadata = {
    "model": "random forest, 200 trees",
    "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "training_rows": len(X_train),
    "features": list(X_train.columns),
    "test_roc_auc": round(roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1]), 3),
    "python": sys.version.split()[0],
    "scikit-learn": version("scikit-learn"),
}
with open("saved_models/churn_pipeline.json", "w") as file:
    json.dump(metadata, file, indent=2)

loaded = joblib.load("saved_models/churn_pipeline.joblib")
print("Same predictions after loading:", (loaded.predict(X_test) == pipeline.predict(X_test)).all())
print(f"File size: {os.path.getsize('saved_models/churn_pipeline.joblib') / 1e6:.1f} MB")

Same predictions after loading: True
File size: 5.2 MB


**Only load files you trust.** `joblib` and `pickle` files can run code when they are loaded, so a file from an unknown source can do anything on your computer. That is why MLflow, in the next section, saves scikit-learn models in the safer **skops** format by default.

## 4. Tracking experiments with MLflow

In Week 5 you compared models in a table you built by hand. After a few days of experiments, nobody remembers which notebook produced which number. **MLflow** records every experiment for you.

### The vocabulary

- An **experiment** groups related runs, for example everything about churn.
- A **run** is one training: one model with one set of settings.
- Each run records **parameters** (the settings you chose, such as `n_estimators`), **metrics** (the results, such as the cross-validated AUC), **tags** (labels such as the author or the dataset version) and **artifacts** (files: the model itself, plots, data samples).

MLflow stores all of this in a **tracking store**. Here it is a SQLite database file next to the notebook:

In [5]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("churn")
print("Tracking to:", mlflow.get_tracking_uri())

Tracking to: sqlite:///mlflow.db


Now three models, each in its own run. For each one we log the parameters and a tag, compute the 5-fold cross-validated AUC on the training data, and log the fitted pipeline as a model.

Two arguments of `log_model` deserve a word. The **signature** records the columns and types the model expects, so that MLflow can refuse wrong inputs later. **`skops_trusted_types`** answers the safety check: skops refuses to save object types it does not know (here, the trees of the forest) unless you say you trust them. We trust exactly the types of the pipeline we just built, and nothing else.

In [6]:
candidates = {
    "logistic_regression": LogisticRegression(max_iter=1000, C=1.0),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "gradient_boosting": HistGradientBoostingClassifier(learning_rate=0.1, random_state=42),
}
logged_models = {}

for name, model in candidates.items():
    candidate = make_pipeline(model)
    with mlflow.start_run(run_name=name) as run:
        mlflow.set_tag("developer", "ana")
        mlflow.set_tag("data", "churn.csv, 80% training split")
        mlflow.log_param("model", name)
        mlflow.log_params({key: value for key, value in model.get_params().items()
                           if key in ("C", "n_estimators", "learning_rate", "max_depth")})
        cv_auc = cross_val_score(candidate, X_train, y_train, cv=5, scoring="roc_auc").mean()
        candidate.fit(X_train, y_train)
        mlflow.log_metric("cv_auc", cv_auc)
        info = mlflow.sklearn.log_model(
            candidate, name="model",
            signature=infer_signature(X_train, candidate.predict_proba(X_train)[:, 1]),
            input_example=X_train.head(3),
            skops_trusted_types=skops.io.get_untrusted_types(data=skops.io.dumps(candidate)),
        )
        logged_models[name] = {"run_id": run.info.run_id, "model_uri": info.model_uri, "model_id": info.model_id}
    print(f"{name:20s} cv_auc = {cv_auc:.3f}")

logistic_regression  cv_auc = 0.816


random_forest        cv_auc = 0.920


gradient_boosting    cv_auc = 0.918


Every run is now stored. `search_runs` returns them as a DataFrame, which you can sort and filter like any other: this is the comparison table, built for you.

In [7]:
runs = mlflow.search_runs(experiment_names=["churn"], order_by=["metrics.cv_auc DESC"])
runs[["tags.mlflow.runName", "params.model", "params.n_estimators", "params.C", "metrics.cv_auc", "tags.developer"]]

,tags.mlflow.runName,params.model,params.n_estimators,params.C,metrics.cv_auc,tags.developer
0,random_forest,random_forest,200,NaN,0.919595,ana
1,gradient_boosting,gradient_boosting,NaN,NaN,0.918106,ana
2,logistic_regression,logistic_regression,NaN,1.0,0.816175,ana


The random forest and gradient boosting are close (about 0.92), and logistic regression is clearly behind (about 0.82), as in Week 5. The same results are easier to explore in the **MLflow UI**. In a terminal, in this folder, run:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

and open http://127.0.0.1:5000 in your browser. You can sort runs, compare them side by side, plot a metric against a parameter, and open each run's artifacts.

[example screenshot of the MLflow UI: the "churn" experiment with three runs, columns for model, n_estimators and cv_auc, and the compare button]

### Where the tracking store lives

| Setup | Where the runs are stored | Fits |
|---|---|---|
| Local folder (the default with no tracking URI) | files in `mlruns/` on your laptop | trying MLflow on your own |
| Local database, as here | a SQLite file and an artifacts folder on your laptop | one person, with the Model Registry |
| Remote tracking server | a server (for example on AWS EC2) with a PostgreSQL database for runs and S3 for artifacts | a team that shares experiments and models |

The code does not change between them: only the tracking URI does.

## 5. The Model Registry

Tracking answers "which run was best?". The **Model Registry** answers "which model do we use?". A **registered model** has a unique name, such as `churn-model`, and contains numbered **versions**, each linked to the run that produced it.

Instead of copying model files around, the application asks the registry for a model by name and **alias**: `models:/churn-model@champion` means "whatever version is the champion right now". Promoting a new model is then a matter of moving the alias, with no change to the application.

Register the best run as version 1 of `churn-model`, and make it the champion:

In [8]:
client = MlflowClient()
best_name = runs.iloc[0]["tags.mlflow.runName"]
best = logged_models[best_name]

client.create_registered_model("churn-model", description="Predicts which telecom customers will churn.")
version_1 = client.create_model_version("churn-model", source=best["model_uri"], run_id=best["run_id"], model_id=best["model_id"])
client.set_registered_model_alias("churn-model", "champion", version_1.version)
print(f"Registered {best_name} as churn-model version {version_1.version}, alias 'champion'")

Registered random_forest as churn-model version 1, alias 'champion'


The application never needs to know which run, file or version that is. It loads the champion:

In [9]:
production_model = mlflow.sklearn.load_model("models:/churn-model@champion")
print(f"Champion test ROC AUC: {roc_auc_score(y_test, production_model.predict_proba(X_test)[:, 1]):.3f}")
production_model.predict_proba(X_test.head(3))[:, 1].round(3)

Champion test ROC AUC: 0.888


array([0.845, 0.03 , 0.11 ])

Now a teammate trains a challenger: gradient boosting with a lower learning rate. It becomes version 2 with the alias `challenger`. The decision rule is agreed in advance: the challenger replaces the champion only if its **cross-validated AUC** is higher. The test set stays out of the decision, so that it remains an honest final check:

In [10]:
challenger_pipeline = make_pipeline(HistGradientBoostingClassifier(learning_rate=0.05, random_state=42))
with mlflow.start_run(run_name="gradient_boosting_slow") as run:
    mlflow.set_tag("developer", "luis")
    mlflow.log_params({"model": "gradient_boosting", "learning_rate": 0.05})
    challenger_cv = cross_val_score(challenger_pipeline, X_train, y_train, cv=5, scoring="roc_auc").mean()
    mlflow.log_metric("cv_auc", challenger_cv)
    challenger_pipeline.fit(X_train, y_train)
    info = mlflow.sklearn.log_model(challenger_pipeline, name="model",
                                    signature=infer_signature(X_train, challenger_pipeline.predict_proba(X_train)[:, 1]),
                                    skops_trusted_types=skops.io.get_untrusted_types(data=skops.io.dumps(challenger_pipeline)))

version_2 = client.create_model_version("churn-model", source=info.model_uri, run_id=run.info.run_id, model_id=info.model_id)
client.set_registered_model_alias("churn-model", "challenger", version_2.version)

champion_cv = runs.iloc[0]["metrics.cv_auc"]
print(f"champion   (version {version_1.version}): cv_auc {champion_cv:.3f}")
print(f"challenger (version {version_2.version}): cv_auc {challenger_cv:.3f}")

if challenger_cv > champion_cv:
    client.set_registered_model_alias("churn-model", "champion", version_2.version)
    print("The challenger wins: version", version_2.version, "is the new champion")
else:
    print("The champion stays: version", version_1.version)

champion   (version 1): cv_auc 0.920
challenger (version 2): cv_auc 0.922
The challenger wins: version 2 is the new champion


In the reference run, the challenger's cross-validated AUC was 0.922 against 0.920, so the rule promoted it. Look at the size of that difference: two thousandths, well within the noise of 5-fold cross-validation. That is why many teams require a **minimum improvement** (for example 0.01) before promoting, or a check on fresh data: every promotion carries a small risk, and it should be worth it.

Promoting is one line, and the application changes nothing: the next time it loads `models:/churn-model@champion`, it gets whichever version holds the alias.

**Stages, the older way.** Before aliases, MLflow used **stages**: each model version was in exactly one stage at a time, *None*, *Staging*, *Production* or *Archived*, and promoting meant moving a version from Staging to Production. Stages are deprecated in current MLflow, but you will still meet them in older projects and documentation. The idea is the same: a name for "the version in use", separate from the version numbers. Note also that logging a model never puts it in production by itself: someone has to register it and choose.

(`production_model`, loaded above, is still version 1: an application that has already loaded a model keeps it until it loads again. We keep using it in section 7.)

## 6. From DEV to PROD

In a team, nobody pushes straight to the version that users depend on. Code moves through **environments**:

- **DEV** (development): where you write and try things, on your own branch.
- **PROD** (production): what users get. In git, usually the `main` branch.

Between them sits a **pull request (PR)**: you propose your branch, a teammate (the **gatekeeper**, or reviewer) reads the changes, **rebuilds your environment from `requirements.txt`**, runs the checks, and only then merges into `main`.

[visual: a developer's branch → pull request → review + checks (environment rebuilt, tests pass) → merge into main (PROD); a red cross on a direct push to main]

```bash
git checkout -b improve-churn-model        # a branch for your change
# ... edit, run, commit ...
git push -u origin improve-churn-model     # then open a pull request on GitHub
```

The checks should be automatic, so that nobody forgets them. Here is a small **model check** of the kind a gatekeeper runs before merging: does the model load, does it accept the expected columns, are the predictions valid probabilities, and is it at least as good as an agreed minimum?

In [11]:
def check_model(model, X_check, y_check, min_auc=0.85):
    probabilities = model.predict_proba(X_check)[:, 1]
    assert list(X_check.columns) == metadata["features"], "the input columns changed"
    assert ((probabilities >= 0) & (probabilities <= 1)).all(), "predictions are not probabilities"
    auc = roc_auc_score(y_check, probabilities)
    assert auc >= min_auc, f"ROC AUC {auc:.3f} is below the agreed minimum of {min_auc}"
    return f"All checks passed (ROC AUC {auc:.3f})"

print(check_model(mlflow.sklearn.load_model("models:/churn-model@champion"), X_test, y_test))

try:
    check_model(make_pipeline(LogisticRegression(max_iter=1000)).fit(X_train, y_train), X_test, y_test)
except AssertionError as error:
    print("Check failed:", error)

All checks passed (ROC AUC 0.904)
Check failed: ROC AUC 0.818 is below the agreed minimum of 0.85


The champion (now version 2) passes; a logistic regression would be blocked by the minimum AUC. In a real repository, a check like this lives in a test file and runs automatically on every pull request with a service such as **GitHub Actions**: this is **continuous integration (CI)**. When merging into `main` also deploys the new version automatically, that is **continuous delivery (CD)**.

## 7. Monitoring and drift

A deployed model is not finished. The world changes: prices, products, customers. When the data the model sees in production stops looking like its training data, that is **data drift**. When the relationship between inputs and outcome changes (a competitor's offer makes heavy users leave, whatever their bill), that is **concept drift**. Either way, the model gets worse **without any change to its code**.

Here is a simulation: next quarter, a new tariff makes everyone use about 30% more daytime minutes, and customers call support more often.

In [12]:
rng = np.random.default_rng(0)
next_quarter = X_test.copy()
next_quarter["Day Mins"] = next_quarter["Day Mins"] * 1.3
next_quarter["Day Charge"] = next_quarter["Day Charge"] * 1.3
next_quarter["CustServ Calls"] = next_quarter["CustServ Calls"] + rng.poisson(1, len(next_quarter))

drift = []
for column in numeric:
    statistic, p_value = ks_2samp(X_train[column], next_quarter[column])
    drift.append({"feature": column, "ks_statistic": round(statistic, 3), "p_value": p_value, "drift": p_value < 0.01})
drift = pd.DataFrame(drift).sort_values("ks_statistic", ascending=False)
drift.head(5)

,feature,ks_statistic,p_value,drift
2,Day Mins,0.356,1.342926e-60,True
4,Day Charge,0.356,1.342926e-60,True
14,CustServ Calls,0.295,1.712858e-41,True
5,Eve Mins,0.056,6.694237e-02,False
7,Eve Charge,0.056,6.694237e-02,False


The **Kolmogorov-Smirnov test** compares the distribution of each feature in the training data with the new data. A large statistic (and a tiny p-value) means the distributions differ. The three features we changed come out on top; the others do not drift. With thousands of rows, even tiny changes become "significant", so in practice teams watch the size of the statistic, not just the p-value.

Drift in the inputs shows up in the predictions too:

In [13]:
current = production_model.predict_proba(X_test)[:, 1]
drifted = production_model.predict_proba(next_quarter)[:, 1]
print(f"Predicted churn rate on this quarter's customers: {(current > 0.5).mean():.1%}")
print(f"Predicted churn rate after the tariff change:     {(drifted > 0.5).mean():.1%}")

Predicted churn rate on this quarter's customers: 10.8%
Predicted churn rate after the tariff change:     29.8%


The model now predicts that far more customers will churn. Maybe they will; or maybe heavy daytime use simply means something different under the new tariff, and the model is wrong. You cannot know until the real outcomes arrive, which is exactly why you **monitor**:

- **The inputs**: distributions of the features, missing values, new categories.
- **The predictions**: their distribution and the share of each class.
- **The performance**, once the true outcomes are known (did those customers actually leave?).

When monitoring raises an alarm, you investigate, and usually **retrain** on recent data, register the new version as a challenger, compare it, and move the alias. Everything from sections 2 to 6 makes that loop fast and safe.

## 8. MLOps responsibly

- **Know what each model was trained on.** Tags, metadata and the registry are what let you answer an auditor, a customer or a regulator. Under the EU AI Act, high-risk systems (credit scoring, for example) must keep technical documentation and logs.
- **Do not log personal data.** Artifacts and input examples are copied into the tracking store, which many people can read. Our input example is three rows of call minutes; never log names, phone numbers or other identifiers (we dropped the `Phone` column).
- **Control who can promote a model.** Moving the `champion` alias changes what every user gets, so it deserves the same review as a merge into `main`.
- **Load only trusted model files**, as in section 3.
- **Monitor fairness, not only accuracy.** Drift can affect one group of customers more than others; break the monitoring down by segment.

## Summary

| Practice | What it gives you | Code |
|---|---|---|
| Virtual environment | a separate Python and libraries per project | `python -m venv .venv`, `source .venv/bin/activate` |
| `requirements.txt` | anyone can rebuild your environment; commit it, never `.venv` | `pip freeze > requirements.txt`, `pip install -r requirements.txt` |
| Save the pipeline | preprocessing and model in one file, plus metadata | `joblib.dump(pipeline, path)`, `joblib.load(path)` |
| Tracking | every run's parameters, metrics, tags and model | `mlflow.start_run()`, `log_param`, `log_metric`, `set_tag`, `mlflow.sklearn.log_model` |
| Compare runs | the comparison table, automatically | `mlflow.search_runs(...)`, `mlflow ui` |
| Registry | named models with versions and aliases | `client.create_model_version(...)`, `set_registered_model_alias(...)` |
| Load the production model | the application asks for a name, not a file | `mlflow.sklearn.load_model("models:/churn-model@champion")` |
| DEV to PROD | branches, pull requests and checks before merging | `git checkout -b ...`, a pull request, CI |
| Monitoring | notice drift before users do | `ks_2samp(train_column, new_column)` |

The rules that matter most:

+ **Commit `requirements.txt`, not `.venv`**, and pin your direct dependencies.
+ **Save the whole pipeline**, with metadata, and load only files you trust.
+ **Track every run**: if it is not tracked, it did not happen.
+ **Applications load models by alias**, and promoting a model is a reviewed decision.
+ **Models decay**: monitor the inputs, the predictions and the performance.

In the lab, you work in pairs: one of you is the developer, the other the gatekeeper, and a model reaches `main` only when the gatekeeper can rebuild the environment and the checks pass. The extra lab goes deeper into MLflow.

## Appendix

### A.1 Autologging

`mlflow.sklearn.autolog()` logs parameters, metrics and the model for every `fit` without any `log_*` calls. It is convenient for quick experiments; explicit logging, as above, gives you control over what is recorded.

In [14]:
warnings.filterwarnings("ignore", category=FutureWarning)       # autolog uses an argument that scikit-learn is renaming
mlflow.sklearn.autolog(log_models=False)
with mlflow.start_run(run_name="autolog_example"):
    make_pipeline(LogisticRegression(max_iter=1000, C=0.1)).fit(X_train, y_train)
mlflow.sklearn.autolog(disable=True)

last = mlflow.search_runs(experiment_names=["churn"], order_by=["start_time DESC"]).iloc[0]
print("Parameters logged automatically:", len([c for c in last.index if c.startswith("params.") and pd.notna(last[c])]))
print("Training metrics:", {c.replace("metrics.", ""): round(float(last[c]), 3) for c in last.index if c.startswith("metrics.training_") and pd.notna(last[c])})

Parameters logged automatically: 40
Training metrics: {'training_precision_score': 0.834, 'training_roc_auc': 0.82, 'training_score': 0.863, 'training_log_loss': 0.325, 'training_f1_score': 0.83, 'training_accuracy_score': 0.863, 'training_recall_score': 0.863}


### A.2 Serving a registered model

MLflow can serve a registered model as a web API in one command, which other applications call over HTTP:

```bash
mlflow models serve -m "models:/churn-model@champion" --port 5001 --env-manager local
```

On Day 3 you will put a model behind a different kind of interface: a Streamlit app.

### A.3 Versioning data

Code is versioned with git and models with the registry, but data changes too. Tools such as **DVC** (Data Version Control) version large datasets next to the code. A simpler habit that already helps: record which file, query or date range each run used, as the `data` tag does above.